# build_advisory_ghsa
`stg_ghsa_normalized`(Advisory 레벨 컬럼만) → `silver_advisory_ghsa` (SCD2, 값이 바뀔 때만 새 버전).
Grain: `ghsa_id` + valid period. `record_hash`는 **내용 컬럼만** 반영한다 — `dq_flags`처럼 파생된 값이나 `batch_id`/`ingested_at`을 넣으면 실행할 때마다, 혹은 결측 플래그가 왔다갔다 할 때마다 불필요한 버전이 생긴다(`silver_kev`에서 이미 겪은 문제, 03_Gold_Schema.md 참고).

In [ ]:
%run 00_silver_common

In [ ]:
norm_df = spark.table(TBL["stg_ghsa_normalized"])

CONTENT_COLS = ["summary", "description", "severity", "published_at", "updated_at",
                 "withdrawn_at", "html_url", "cwe_ids", "cvss_score", "cvss_vector"]

advisory_df = (
    norm_df
    .withColumn("record_hash", content_hash([F.col(c) for c in CONTENT_COLS]))
    .select(
        "ghsa_id", "summary", "description", "severity",
        "published_at", "updated_at", "withdrawn_at", "html_url",
        "cwe_ids", "cvss_score", "cvss_vector", "dq_flags",
        "valid_from", "source", "batch_id", "ingested_at", "record_hash",
    )
)


In [ ]:
from pyspark.sql import Window

# 같은 ghsa_id가 이 batch 안에서 여러 번 나오면(중복 수집) valid_from이 가장 최신인 것만 남긴다.
advisory_df = (
    advisory_df
    .withColumn("_rn", F.row_number().over(Window.partitionBy("ghsa_id").orderBy(F.col("valid_from").desc())))
    .filter(F.col("_rn") == 1)
    .drop("_rn")
)

result = merge_scd2(advisory_df, TBL["advisory_ghsa"], key_cols=["ghsa_id"])

log_batch_run("build_advisory_ghsa", "SUCCESS", record_count=result.get("opened", 0))
log_dq_metric("build_advisory_ghsa", "versions_opened", result.get("opened", 0), passed=True)
print(f"[build_advisory_ghsa] {result} batch_id={batch_id}")
